# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [1]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [2]:
# Initialization

load_dotenv(override=True)

openai_api_key = os.getenv('OPENAI_API_KEY')
if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
MODEL = "gpt-4.1-mini"
openai = OpenAI()

# As an alternative, if you'd like to use Ollama instead of OpenAI
# Check that Ollama is running for you locally (see week1/day2 exercise) then uncomment these next 2 lines
# MODEL = "llama3.2"
# openai = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')


OpenAI API Key exists and begins sk-proj-


In [3]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [4]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


## Tools

Tools are an incredibly powerful feature provided by the frontier LLMs.

With tools, you can write a function, and have the LLM call that function as part of its response.

Sounds almost spooky.. we're giving it the power to run code on our machine?

Well, kinda.

In [5]:
# Let's start by making a useful function

ticket_prices = {"london": "$799", "paris": "$899", "tokyo": "$1400", "berlin": "$499"}

def get_ticket_price(destination_city):
    print(f"Tool called for city {destination_city}")
    price = ticket_prices.get(destination_city.lower(), "Unknown ticket price")
    return f"The price of a ticket to {destination_city} is {price}"


In [6]:
get_ticket_price("London")

Tool called for city London


'The price of a ticket to London is $799'

In [7]:
# There's a particular dictionary structure that's required to describe our function:

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [8]:
# And this is included in a list of tools:

tools = [{"type": "function", "function": price_function}]

In [9]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

## Getting OpenAI to use our Tool

There's some fiddly stuff to allow OpenAI "to call our tool"

What we actually do is give the LLM the opportunity to inform us that it wants us to run the tool.

Here's how the new chat function looks:

In [13]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.append(response)

        for message in messages:
            print(message)
            
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [11]:
# We have to write that function handle_tool_call:

def handle_tool_call(message):
    tool_call = message.tool_calls[0]
    if tool_call.function.name == "get_ticket_price":
        arguments = json.loads(tool_call.function.arguments)
        city = arguments.get('destination_city')
        price_details = get_ticket_price(city)
        response = {
            "role": "tool",
            "content": price_details,
            "tool_call_id": tool_call.id
        }
    return response

In [14]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


Tool called for city tokyo
{'role': 'system', 'content': "\nYou are a helpful assistant for an Airline called FlightAI.\nGive short, courteous answers, no more than 1 sentence.\nAlways be accurate. If you don't know the answer, say so.\n"}
{'role': 'user', 'content': 'hello'}
{'role': 'assistant', 'content': 'Hello! How can I assist you with your flight today?'}
{'role': 'user', 'content': 'I would liek to book a trip'}
{'role': 'assistant', 'content': 'Sure! Could you please tell me the destination city for your trip?'}
{'role': 'user', 'content': 'tokyo please'}
ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_3dev0d8ZUqkR6khaoKnNuUoi', function=Function(arguments='{"destination_city":"tokyo"}', name='get_ticket_price'), type='function')])
{'role': 'tool', 'content': 'The price of a ticket to tokyo is $1400', 'tool_call_id': 'call_3dev0d8ZUqkR6khaoKnNuUoi'}


## Let's make a couple of improvements

Handling multiple tool calls in 1 response

Handling multiple tool calls 1 after another

In [15]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    if response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages)
    
    return response.choices[0].message.content

In [16]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7875
* To create a public link, set `share=True` in `launch()`.


Tool called for city Paris
Tool called for city Tokyo
Tool called for city London
Tool called for city Paris


In [18]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = openai.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [19]:
import sqlite3


In [20]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)')
    conn.commit()

In [21]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [22]:
get_ticket_price("London")

DATABASE TOOL CALLED: Getting price for London


'No price data available for this city'

In [23]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?', (city.lower(), price, price))
        conn.commit()

In [24]:
ticket_prices = {"london":799, "paris": 899, "tokyo": 1420, "sydney": 2999}
for city, price in ticket_prices.items():
    set_ticket_price(city, price)

In [25]:
get_ticket_price("Tokyo")

DATABASE TOOL CALLED: Getting price for Tokyo


'Ticket price to Tokyo is $1420.0'

In [26]:
gr.ChatInterface(fn=chat, type="messages").launch()

* Running on local URL:  http://127.0.0.1:7884
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for Tokyo


## Exercise

Add a tool to set the price of a ticket!

In [ ]:
# ============================================================================
# COMPLETE AIRLINE ASSISTANT - Production-Style Reference
# ============================================================================
# Future me: This is the complete implementation with all improvements.
# Key learnings I wanted to capture:
# - Function registry pattern (no if statements - scales beautifully)
# - SQLite for conversation history (persistent, queryable, production-ready)
# - Proper session management using Gradio's state
# - Complete error handling
# - Streaming + tool calls working together
# - All tools implemented (get and set prices)
# ============================================================================

import os
import json
import sqlite3
import uuid
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
from typing import Dict, Callable, List, Any, Optional

# Load environment
load_dotenv(override=True)
MODEL = "gpt-4.1-mini"
openai = OpenAI()

# Database setup
DB_PRICES = "prices.db"
DB_CONVERSATIONS = "conversations.db"

# Initialize prices database
# Simple key-value store.
with sqlite3.connect(DB_PRICES) as conn:
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS prices (
            city TEXT PRIMARY KEY, 
            price REAL
        )
    ''')
    conn.commit()

# Initialize conversations database
# This stores ALL conversation history. Each message gets saved here.
# session_id tracks separate conversations; tool_calls are stored as JSON to reconstruct assistant state

with sqlite3.connect(DB_CONVERSATIONS) as conn:
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS conversations (
            id INTEGER PRIMARY KEY AUTOINCREMENT,
            session_id TEXT NOT NULL,
            role TEXT NOT NULL,
            content TEXT,
            tool_calls TEXT,
            timestamp DATETIME DEFAULT CURRENT_TIMESTAMP
        )
    ''')
    # Create index separately (SQLite doesn't allow INDEX in CREATE TABLE)
    cursor.execute('''
        CREATE INDEX IF NOT EXISTS idx_session ON conversations(session_id)
    ''')
    conn.commit()

# System message
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
You can get ticket prices and set ticket prices using the available tools.
"""

# ============================================================================
# TOOL FUNCTIONS
# ============================================================================

def get_ticket_price(destination_city: str) -> str:
    """
    Get ticket price from database.
    """
    print(f"TOOL: get_ticket_price({destination_city})", flush=True)
    try:
        with sqlite3.connect(DB_PRICES) as conn:
            cursor = conn.cursor()
            cursor.execute('SELECT price FROM prices WHERE city = ?', (destination_city.lower(),))
            result = cursor.fetchone()
            if result:
                return f"Ticket price to {destination_city} is ${result[0]:.2f}"
            return f"No price data available for {destination_city}"
    except Exception as e:
        return f"Error retrieving price: {str(e)}"

def set_ticket_price(destination_city: str, price: float) -> str:
    """
    Set ticket price in database.
    """
    print(f"TOOL: set_ticket_price({destination_city}, ${price})", flush=True)
    try:
        # Validate price
        if price <= 0:
            return f"Error: Price must be positive. Received ${price:.2f}"
        
        if not destination_city or not destination_city.strip():
            return "Error: Destination city cannot be empty"
        
        with sqlite3.connect(DB_PRICES) as conn:
            cursor = conn.cursor()
            cursor.execute(
                'INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?',
                (destination_city.lower().strip(), price, price)
            )
            conn.commit()
        return f"Successfully set ticket price to {destination_city} as ${price:.2f}"
    except ValueError as e:
        return f"Error: Invalid price format. {str(e)}"
    except Exception as e:
        return f"Error setting price: {str(e)}"

# ============================================================================
# FUNCTION REGISTRY PATTERN - This is the key improvement!
# ============================================================================
# Instead of if/elif chains, we use a dictionary lookup.
# When OpenAI calls a tool, we look it up here and execute it directly.
# To add a new tool: just add the function to this dict. 
# This scales much better as tools grow


TOOL_REGISTRY: Dict[str, Callable] = {
    "get_ticket_price": get_ticket_price,
    "set_ticket_price": set_ticket_price,
}

# ============================================================================
# TOOL DEFINITIONS - JSON Schema for OpenAI
# ============================================================================
# OpenAI needs to know what tools are available and their schemas.
# This is the "contract" - it tells OpenAI what parameters each tool expects.
# The function name here MUST match the key in TOOL_REGISTRY above.

get_price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to a destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}


set_price_function = {
    "name": "set_ticket_price",
    "description": "Set or update the price of a return ticket to a destination city. Use this when the customer or admin wants to change a price.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city to set the price for",
            },
            "price": {
                "type": "number",
                "description": "The price in dollars (e.g., 799.99)",
            },
        },
        "required": ["destination_city", "price"],
        "additionalProperties": False
    }
}

tools = [
    {"type": "function", "function": get_price_function},
    {"type": "function", "function": set_price_function},
]

# ============================================================================
# CONVERSATION HISTORY MANAGEMENT (SQLite)
# ============================================================================

def save_message(session_id: str, role: str, content: str, tool_calls: Optional[str] = None):
    """
    Save a message to the conversation database.
    Every message (user, assistant, system, tool) gets saved here.
    tool_calls is JSON string for assistant messages that called tools.
    """
    with sqlite3.connect(DB_CONVERSATIONS) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'INSERT INTO conversations (session_id, role, content, tool_calls) VALUES (?, ?, ?, ?)',
            (session_id, role, content, tool_calls)
        )
        conn.commit()

def load_conversation(session_id: str) -> List[Dict[str, Any]]:
    """
    Load conversation history from database.
    This reconstructs the full conversation for the API call.
    We parse tool_calls JSON to properly reconstruct assistant messages.
    The API needs the full history to maintain context.
    """
    with sqlite3.connect(DB_CONVERSATIONS) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'SELECT role, content, tool_calls FROM conversations WHERE session_id = ? ORDER BY id',
            (session_id,)
        )
        rows = cursor.fetchall()
        
        messages = []
        for role, content, tool_calls_json in rows:
            msg = {"role": role, "content": content or ""}
            
            # Parse tool_calls if this was an assistant message with tool calls
            # When assistant calls tools, we need to include the tool_calls
            # in the message so the API knows what happened. This is crucial for
            # maintaining conversation context when tools are involved.
            if role == "assistant" and tool_calls_json:
                try:
                    tool_calls_data = json.loads(tool_calls_json)
                    # Convert back to the format OpenAI expects
                    msg["tool_calls"] = tool_calls_data
                except json.JSONDecodeError:
                    # If parsing fails, just continue without tool_calls
                    pass
            
          # Tool messages must include tool_call_id; OpenAI requires this for tool continuity
            elif role == "tool" and tool_calls_json:
                # For tool messages, tool_calls_json contains the tool_call_id string
                msg["tool_call_id"] = tool_calls_json
            
            messages.append(msg)
        
        return messages

# ============================================================================
# TOOL CALL HANDLER (Using Registry Pattern)
# ============================================================================

def handle_tool_calls(message) -> List[Dict[str, Any]]:
    """
    Handle tool calls using the function registry.
    This is where the magic happens.
    We look up the function in TOOL_REGISTRY and call it with the arguments.
    The **arguments unpacks the dict into keyword arguments.
    """
    responses = []
    
    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        try:
            arguments = json.loads(tool_call.function.arguments)
        except json.JSONDecodeError as e:
            responses.append({
                "role": "tool",
                "content": f"Error: Invalid tool arguments JSON. {str(e)}",
                "tool_call_id": tool_call.id
            })
            continue
        
        # Look up function in registry
        if function_name in TOOL_REGISTRY:
            func = TOOL_REGISTRY[function_name]
            try:
                # Call the function with unpacked arguments
                result = func(**arguments)
                responses.append({
                    "role": "tool",
                    "content": result,
                    "tool_call_id": tool_call.id
                })
            except TypeError as e:
                # Wrong number of arguments or wrong argument names
                responses.append({
                    "role": "tool",
                    "content": f"Error: Function {function_name} received invalid arguments. {str(e)}",
                    "tool_call_id": tool_call.id
                })
            except Exception as e:
                # Any other error from the function itself
                responses.append({
                    "role": "tool",
                    "content": f"Error executing {function_name}: {str(e)}",
                    "tool_call_id": tool_call.id
                })
        else:
            # Tool not found in registry - this shouldn't happen if schemas match
            responses.append({
                "role": "tool",
                "content": f"Error: Unknown tool '{function_name}'. Available tools: {list(TOOL_REGISTRY.keys())}",
                "tool_call_id": tool_call.id
            })
    
    return responses

# ============================================================================
# CHAT FUNCTION WITH STREAMING & SQLITE HISTORY
# ============================================================================

def chat(message: str, history, session_id: str):
    """
    Main chat function with all the improvements.
    This is complex because streaming + tool calls don't play well together.
    Here's the flow:
    1. Load conversation from SQLite (not Gradio's history)
    2. Try streaming first
    3. If tool calls detected, switch to non-streaming to get full tool call data
    4. Execute tools, add responses to conversation
    5. Continue until no more tool calls
    6. Stream final response
    7. Save everything to SQLite
    
    The tricky part: OpenAI's streaming API doesn't give us complete tool call info
    in the stream, so we have to make a non-streaming call when tools are involved.
    """
    # Load conversation from database
    messages = load_conversation(session_id)
    
    # Add system message if this is the first message
    if not messages:
        messages = [{"role": "system", "content": system_message}]
        save_message(session_id, "system", system_message)
    
    # Add user message
    messages.append({"role": "user", "content": message})
    save_message(session_id, "user", message)
    
    # Initial API call
    response = openai.chat.completions.create(
        model=MODEL, 
        messages=messages, 
        tools=tools,
        stream=True  # Enable streaming
    )
    
    # Check if we have tool calls in the stream
    # The streaming API gives us tool calls incrementally, but we need
    # the complete tool call info to execute them. So we detect if tools are coming,
    # then switch to non-streaming to get the full data.
    accumulated_content = ""
    tool_calls_detected = False
    finish_reason = None
    
    # Try to stream, but watch for tool calls
    for chunk in response:
        if chunk.choices[0].delta.content:
            accumulated_content += chunk.choices[0].delta.content
            yield accumulated_content
        
        # Check if this chunk has tool calls
        if chunk.choices[0].delta.tool_calls:
            tool_calls_detected = True
            # Can't continue streaming - need full tool call data
        
        # Track finish_reason from the last chunk
        if chunk.choices[0].finish_reason:
            finish_reason = chunk.choices[0].finish_reason
    
    # If tool calls detected OR no content (meaning it's all tool calls) OR finish_reason is tool_calls,
    # switch to non-streaming to get complete tool call info
    if tool_calls_detected or not accumulated_content or finish_reason == "tool_calls":
        # Make a non-streaming call to get complete response with tool calls
        response = openai.chat.completions.create(
            model=MODEL,
            messages=messages,
            tools=tools
        )
        
        # Handle tool calls loop
        # We loop while we have tool calls. Each iteration:
        # 1. Process tool calls from the current response (ChatCompletion)
        # 2. Stream the next response
        # 3. If more tool calls detected, switch back to non-streaming and continue loop
        # 4. If no more tool calls, break and we're done
        # Note: response starts as ChatCompletion, but becomes Stream inside the loop
        while response.choices[0].finish_reason == "tool_calls":
            assistant_message = response.choices[0].message
            
            # Save assistant message with tool calls
            tool_calls_json = json.dumps([tc.model_dump() for tc in assistant_message.tool_calls]) if assistant_message.tool_calls else None
            save_message(session_id, "assistant", assistant_message.content or "", tool_calls_json)
            messages.append(assistant_message)
            
            # Handle tool calls
            tool_responses = handle_tool_calls(assistant_message)
            messages.extend(tool_responses)
            
            # Save tool responses
            for tool_resp in tool_responses:
                save_message(session_id, "tool", tool_resp["content"], tool_resp.get("tool_call_id"))
            
            # Continue conversation - might need more tool calls
            response = openai.chat.completions.create(
                model=MODEL,
                messages=messages,
                tools=tools,
                stream=True  # Try streaming again for final response
            )
            
            # Stream the response after tool execution
            # We need to track finish_reason from the stream chunks
            # because the Stream object doesn't have finish_reason directly
            accumulated_content = ""
            finish_reason = None
            more_tool_calls_detected = False
            
            for chunk in response:
                if chunk.choices[0].delta.content:
                    accumulated_content += chunk.choices[0].delta.content
                    yield accumulated_content
                
                # Check if more tool calls (rare but possible)
                if chunk.choices[0].delta.tool_calls:
                    more_tool_calls_detected = True
                    # Need to switch back to non-streaming
                    break
                
                # Track finish_reason from the last chunk
                if chunk.choices[0].finish_reason:
                    finish_reason = chunk.choices[0].finish_reason
            
            # If we detected more tool calls OR finish_reason is tool_calls, continue the loop
            if more_tool_calls_detected or finish_reason == "tool_calls":
                # Switch to non-streaming for tool call handling
                response = openai.chat.completions.create(
                    model=MODEL,
                    messages=messages,
                    tools=tools
                )
            else:
                # No more tool calls, we're done - save the final content
                # If we broke out here, we already streamed and saved the content
                # No need to access response.choices since response is still a Stream
                if accumulated_content:
                    save_message(session_id, "assistant", accumulated_content)
                break
        
        # Save final assistant response (only if response is a ChatCompletion, not a Stream)
        # After the while loop, if we broke in the else branch, we already saved.
        # If the while loop exited because finish_reason != "tool_calls", response is a ChatCompletion.
        # But actually, if we broke in else, response is still a Stream, so we check the type.
        try:
            # Try to access as ChatCompletion - if it works, save it
            # If response is a Stream, this will raise AttributeError
            if hasattr(response, 'choices') and response.choices:
                final_content = response.choices[0].message.content
                if final_content:
                    save_message(session_id, "assistant", final_content)
        except (AttributeError, TypeError):
            # Response is a Stream or already consumed - we already saved in the else branch
            pass
    else:
        # Save the streamed content
        if accumulated_content:
            save_message(session_id, "assistant", accumulated_content)

# ============================================================================
# GRADIO INTERFACE - With Session Management
# ============================================================================

def chat_wrapper(message, history, request: gr.Request = None):
    """
    Wrapper to adapt Gradio's history format to our SQLite system.
    Gradio gives us history, but we ignore it and use SQLite instead.
    We try to use Gradio's request object to get a unique session ID per user/browser.
    If request is not available, we generate a unique ID.
    This way each user gets their own conversation thread.
    """
    # Get or create session ID from Gradio's request
    # Request might be None in some Gradio versions or contexts
    # We handle both cases gracefully
    try:
        if request and hasattr(request, 'headers') and request.headers:
            # Try to get a session identifier from headers
            user_agent = request.headers.get('user-agent', '')
            # Create a hash-like identifier (simplified - in production use proper session management)
            session_id = f"session_{hash(user_agent) % 1000000}"
        else:
            # Fallback: generate a unique ID
            session_id = f"session_{uuid.uuid4().hex[:8]}"
    except Exception:
        # If anything goes wrong with request handling, just use a unique ID
        session_id = f"session_{uuid.uuid4().hex[:8]}"
    
    # Use our chat function with streaming
    for response in chat(message, history, session_id):
        yield response

# Launch the interface
# The title helps me remember this is the improved version
gr.ChatInterface(
    fn=chat_wrapper,
    type="messages",
    title="FlightAI Assistant (Complete - Registry + SQLite + Streaming)"
).launch()


* Running on local URL:  http://127.0.0.1:7874
* To create a public link, set `share=True` in `launch()`.


TOOL: get_ticket_price(New York)
TOOL: get_ticket_price(London)
TOOL: set_ticket_price(London, $850)
TOOL: get_ticket_price(London)
TOOL: get_ticket_price(Mars)
TOOL: get_ticket_price(Lisbon)
TOOL: set_ticket_price(Lisbon, $500)
TOOL: get_ticket_price(Paris)
TOOL: set_ticket_price(Paris, $950)
TOOL: get_ticket_price(Paris)
TOOL: get_ticket_price(Paris)
TOOL: set_ticket_price(Paris, $105)
TOOL: get_ticket_price(Paris)


<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business Applications</h2>
            <span style="color:#181;">Hopefully this hardly needs to be stated! You now have the ability to give actions to your LLMs. This Airline Assistant can now do more than answer questions - it could interact with booking APIs to make bookings!</span>
        </td>
    </tr>
</table>